### Dylan Renard
### Applied Machine Learning JHU
### Homework 7
### Professor Alhassan
### March 9th, 2026


## HW Assignment 7: Applied Machine Learning @JHU

#### Continuing  with the previous  machine learning  problem,  let's get back to the pre-processed dataset  Suicide Rates Overview 1985 to 2016 file. We would  like to have a machine learning model to predict the suicide  rate 'suicides/100k pop'.  

#### Q1 [10 pts] Use your previous pre-processed  dataset,  keep the variables  as one-hot  encoded, and develop  a multiple  linear  regression  model.  How many regression  coefficients does this model have? 

In [1]:
# Import required libraries
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# ----------------------------------------------------------
# Step 1: Load the dataset used in the previous assignment
# ----------------------------------------------------------

# The dataset contains suicide statistics for different
# demographic groups from 1985–2016.
df = pd.read_csv("master.csv")

# ----------------------------------------------------------
# Step 2: Select the variables used for the regression model
# ----------------------------------------------------------

# Predictor variables (categorical)
#   sex
#   age
#   generation
#
# Target variable
#   suicides/100k pop

features = ['sex', 'age', 'generation']
target = 'suicides/100k pop'

X = df[features]
y = df[target]

# ----------------------------------------------------------
# Step 3: One-hot encode the categorical predictors
# ----------------------------------------------------------

# One-hot encoding converts each category into a binary column.
# This matches the preprocessing done in Module 3.

X_encoded = pd.get_dummies(X)

# Example columns produced:
#   sex_female
#   sex_male
#   age_15-24 years
#   age_25-34 years
#   ...
#   generation_Generation X
#   generation_Millenials
#   etc.

# ----------------------------------------------------------
# Step 4: Train a multiple linear regression model
# ----------------------------------------------------------

model = LinearRegression()

# Fit the regression model using the encoded predictors
model.fit(X_encoded, y)

# ----------------------------------------------------------
# Step 5: Determine the number of regression coefficients
# ----------------------------------------------------------

# Each feature column receives a regression coefficient.
num_coefficients = len(model.coef_)

print("Number of regression coefficients:", num_coefficients)

# Optional: display feature names with coefficients
coef_table = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_
})

print(coef_table)

# Also show the intercept term of the regression model
print("Intercept:", model.intercept_)

Number of regression coefficients: 14
                       Feature  Coefficient
0                   sex_female    -7.423231
1                     sex_male     7.423231
2              age_15-24 years    -2.887394
3              age_25-34 years    -0.224249
4              age_35-54 years     1.933153
5               age_5-14 years   -10.967285
6              age_55-74 years     2.628253
7                age_75+ years     9.517523
8           generation_Boomers     0.245961
9   generation_G.I. Generation     3.060943
10     generation_Generation X    -0.564352
11     generation_Generation Z    -1.340047
12       generation_Millenials    -1.575624
13           generation_Silent     0.173119
Intercept: 12.949631859527006


#### Q1 Interpretation:
A multiple linear regression model was trained using the one-hot encoded predictor variables sex, age, and generation to predict the target variable suicides/100k pop. After one-hot encoding, the categorical variables produced 14 feature columns, resulting in 14 regression coefficients in the model. Specifically, the variables included 2 coefficients for sex, 6 coefficients for age groups, and 6 coefficients for generation groups. In addition to these coefficients, the model includes an intercept term of approximately 12.95, which represents the baseline predicted suicide rate when all feature indicators are zero.

Examining the coefficients shows meaningful patterns. For example, the coefficient for sex_male (+7.42) is positive while sex_female (−7.42) is negative, suggesting higher predicted suicide rates for males relative to females. Age group coefficients also indicate increasing suicide rates for older age groups, with the 75+ years category having the largest positive coefficient. These coefficients reflect how each demographic category contributes to the predicted suicide rate in the regression model.

#### Q2 [10 pts] Use this  model to predict  the target variable  for people  with age 20, male, and generation  X. Report  this prediction.  What  is the MAE error of this prediction?  

In [2]:
# ----------------------------------------------------------
# Q2: Predict for age 20, male, Generation X
# ----------------------------------------------------------

# The dataset stores age as grouped categories, not exact ages.
# Since age 20 falls in the 15-24 years category, we represent
# the new person using the same categorical structure as the
# original dataset.

new_person = pd.DataFrame({
    'sex': ['male'],
    'age': ['15-24 years'],
    'generation': ['Generation X']
})

# One-hot encode the new observation using the same approach
# as the training predictors
new_person_encoded = pd.get_dummies(new_person)

# Reindex the encoded row so it matches the exact columns
# used in the trained regression model. Any missing columns
# are filled with 0.
new_person_encoded = new_person_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Predict the suicide rate for this person
q2_prediction = model.predict(new_person_encoded)[0]
print("Predicted suicides/100k pop for age 20, male, Generation X:", q2_prediction)

# ----------------------------------------------------------
# Compute MAE for the Q1 regression model
# ----------------------------------------------------------

# Predict the target values for the full dataset
y_pred = model.predict(X_encoded)

# Compute mean absolute error across all observations
q2_mae = mean_absolute_error(y, y_pred)
print("MAE of Q1 regression model:", q2_mae)

Predicted suicides/100k pop for age 20, male, Generation X: 16.921116488208327
MAE of Q1 regression model: 10.180346495350713


#### Q2 Interpretation:
Using the multiple linear regression model from Q1, the predicted suicide rate for a 20-year-old male from Generation X is 16.92 suicides per 100,000 population. Since the dataset represents age in grouped categories, age 20 was mapped to the 15–24 years age group before applying one-hot encoding. The encoded observation activated the variables sex_male, age_15–24 years, and generation_Generation X, while all other feature indicators were set to zero.

The Mean Absolute Error (MAE) of the regression model across the dataset is 10.18, meaning that on average the model’s predictions differ from the observed suicide rates by approximately 10.18 suicides per 100,000 population. This indicates that while the model captures some demographic trends in the data, there is still substantial prediction error, suggesting that additional variables or a different model structure may be needed to improve predictive accuracy.

#### Q3 [20 pts] Now go back to the original  sex, age,  and generation  variables  in their original numerical form (i.e.  prior to the one-hot  encoding and  build  a new model. I.e., feature engineer  the original  nominal  age and generation  features  into truly numerical  features.) How many line  coefficients are there?  

In [3]:
# ----------------------------------------------------------
# Q3: Build a new regression model using numerical features
# instead of one-hot encoded categorical variables
# ----------------------------------------------------------

import pandas as pd
from sklearn.linear_model import LinearRegression

# ----------------------------------------------------------
# Step 1: Load the dataset
# ----------------------------------------------------------

df = pd.read_csv("master.csv")

# Clean column names to remove leading/trailing whitespace
df.columns = df.columns.str.strip()

# Convert GDP-for-year from string with commas to numeric
df['gdp_for_year ($)'] = (
    df['gdp_for_year ($)']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# ----------------------------------------------------------
# Step 2: Create a working copy for Q3
# ----------------------------------------------------------

df_q3 = df.copy()

# ----------------------------------------------------------
# Step 3: Feature engineer sex into a numeric variable
# ----------------------------------------------------------
# female = 0, male = 1

df_q3['sex_numeric'] = df_q3['sex'].map({
    'female': 0,
    'male': 1
})

# ----------------------------------------------------------
# Step 4: Feature engineer age groups into truly numerical values
# ----------------------------------------------------------
# Use approximate midpoints for each age range

age_map = {
    '5-14 years': 10,
    '15-24 years': 20,
    '25-34 years': 30,
    '35-54 years': 45,
    '55-74 years': 65,
    '75+ years': 80
}

df_q3['age_numeric'] = df_q3['age'].map(age_map)

# ----------------------------------------------------------
# Step 5: Feature engineer generation into a numeric variable
# ----------------------------------------------------------
# Earlier generations get lower values, later generations get higher values

generation_map = {
    'G.I. Generation': 1,
    'Silent': 2,
    'Boomers': 3,
    'Generation X': 4,
    'Millenials': 5,
    'Generation Z': 6
}

df_q3['generation_numeric'] = df_q3['generation'].map(generation_map)

# ----------------------------------------------------------
# Step 6: Select all numeric predictors for the new regression model
# ----------------------------------------------------------
# Keep the original numeric variables in addition to the newly
# engineered numeric versions of sex, age, and generation

features_q3 = [
    'sex_numeric',
    'age_numeric',
    'generation_numeric',
    'year',
    'population',
    'gdp_for_year ($)',
    'gdp_per_capita ($)'
]

target = 'suicides/100k pop'

X_q3 = df_q3[features_q3]
y_q3 = df_q3[target]

# ----------------------------------------------------------
# Step 7: Train the multiple linear regression model
# ----------------------------------------------------------

model_q3 = LinearRegression()
model_q3.fit(X_q3, y_q3)

# ----------------------------------------------------------
# Step 8: Report the number of regression coefficients
# ----------------------------------------------------------

num_coefficients_q3 = len(model_q3.coef_)
print("Number of regression coefficients:", num_coefficients_q3)

# ----------------------------------------------------------
# Step 9: Print coefficient table for interpretation
# ----------------------------------------------------------

coef_table_q3 = pd.DataFrame({
    "Feature": X_q3.columns,
    "Coefficient": model_q3.coef_
})

print(coef_table_q3)

# ----------------------------------------------------------
# Step 10: Print the intercept
# ----------------------------------------------------------

print("Intercept:", model_q3.intercept_)

Number of regression coefficients: 7
              Feature   Coefficient
0         sex_numeric -9.796213e-06
1         age_numeric  2.732555e-01
2  generation_numeric -1.443840e-02
3                year -5.651088e-05
4          population  1.598021e-07
5    gdp_for_year ($)  3.233187e-14
6  gdp_per_capita ($) -1.770922e-06
Intercept: 1.3031003650828605


#### Q3 Interpretation
A new multiple linear regression model was constructed using feature-engineered numeric versions of the original variables sex, age, and generation, along with the existing numerical predictors year, population, gdp_for_year, and gdp_per_capita. The categorical variable sex was converted to a binary numeric variable (female = 0, male = 1), the age groups were mapped to approximate numeric midpoints of each age range, and generation categories were converted to ordinal numeric values representing generational order.

Because the model now uses seven numeric predictors, the regression model contains seven linear coefficients, in addition to the intercept term (approximately 1.303). The coefficient values indicate the linear contribution of each predictor to the predicted suicide rate. For example, the positive coefficient for age_numeric (≈ 0.273) suggests that predicted suicide rates increase with age, while the small negative coefficient for generation_numeric suggests a slight decreasing trend in predicted suicide rates across newer generations. The coefficients for economic variables such as GDP and population are very small due to the large scale of those variables, indicating their effect per unit change is relatively minor within the model.

#### Q4 [10 pts] Use this  new Q3. model to predict  the target value for the people  with age 20, male, and generation  X. Report  the prediction.  What is the MAE error of this  prediction?

In [4]:
# Create a new observation using the same numeric feature engineering
# as the Q3 model

new_person_q4 = pd.DataFrame({
    'sex_numeric': [1],          # male = 1
    'age_numeric': [20],         # age 20
    'generation_numeric': [4],   # Generation X = 4
    'year': [df_q3['year'].mean()],
    'population': [df_q3['population'].mean()],
    'gdp_for_year ($)': [df_q3['gdp_for_year ($)'].mean()],
    'gdp_per_capita ($)': [df_q3['gdp_per_capita ($)'].mean()]
})

# Predict the suicide rate for this person
q4_prediction = model_q3.predict(new_person_q4)[0]
print("Predicted suicides/100k pop for age 20, male, Generation X:", q4_prediction)

# ----------------------------------------------------------
# Compute MAE for the Q3 regression model
# ----------------------------------------------------------

y_pred_q3 = model_q3.predict(X_q3)
q4_mae = mean_absolute_error(y_q3, y_pred_q3)
print("MAE of Q3 regression model:", q4_mae)

Predicted suicides/100k pop for age 20, male, Generation X: 6.876693758388764
MAE of Q3 regression model: 11.290586259833535


#### Q4 Interpretation
Using the regression model developed in Q3, the predicted suicide rate for a 20-year-old male from Generation X is approximately 6.88 suicides per 100,000 population. In this prediction, the engineered numerical features were set as sex_numeric = 1 (male), age_numeric = 20, and generation_numeric = 4 (Generation X). The remaining numerical predictors (year, population, gdp_for_year, and gdp_per_capita) were set to their mean values from the dataset to provide reasonable baseline conditions for the prediction.

The Mean Absolute Error (MAE) of the Q3 regression model is 11.29, meaning that on average the model’s predictions differ from the observed suicide rates by about 11.29 suicides per 100,000 population. Compared to the MAE from the Q1 model, this slightly larger error suggests that converting categorical variables into simple numeric representations may reduce the model’s ability to capture more complex relationships in the data.

#### Q5: [10 pts] Did you note  any change in these  two model  performances? 

Yes, there was a clear difference in the performance of the two regression models. The Q1 model, which used one-hot encoding for the categorical variables, predicted a suicide rate of 16.92 suicides per 100,000 population for a 20-year-old male from Generation X and had a Mean Absolute Error (MAE) of 10.18. The Q3 model, which converted sex, age, and generation into single numeric variables, predicted a lower value of 6.88 suicides per 100,000 population and had a slightly higher MAE of 11.29.

This change in performance likely comes from how the categorical variables were represented in the two models. In the Q1 model, one-hot encoding allows the regression model to learn separate coefficients for each category, which gives the model more flexibility to capture differences between demographic groups. In contrast, the Q3 model forces age and generation into single numeric variables, which assumes that their effects change in a simple linear way across categories. This simplifies the model but can also make it harder to capture more complex patterns in the data. As discussed in the regression chapter of the textbook, the way features are represented—especially categorical variables—can significantly affect how well linear models capture relationships between predictors and the target variable (Raschka, 2026, Ch. 9).

#### Q6: [10 pts] Use your Q3. model to predict the target value for age 33, male, and generation Alpha  (i.e. the generation  after generation  Z); report the prediction

In [5]:
# ----------------------------------------------------------
# Q6: Predict for age 33, male, Generation Alpha
# ----------------------------------------------------------

new_person_q6 = pd.DataFrame({
    'sex_numeric': [1],           # male
    'age_numeric': [33],          # age 33
    'generation_numeric': [7],    # Generation Alpha (next after Gen Z)
    'year': [df_q3['year'].mean()],
    'population': [df_q3['population'].mean()],
    'gdp_for_year ($)': [df_q3['gdp_for_year ($)'].mean()],
    'gdp_per_capita ($)': [df_q3['gdp_per_capita ($)'].mean()]
})

q6_prediction = model_q3.predict(new_person_q6)[0]

print("Predicted suicides/100k pop for age 33, male, Generation Alpha:", q6_prediction)

Predicted suicides/100k pop for age 33, male, Generation Alpha: 10.385700437312602


#### Q6 Interpretation
Using the Q3 regression model, the predicted suicide rate for a 33-year-old male from Generation Alpha is approximately 10.39 suicides per 100,000 population. In this prediction, the engineered numeric features were set as sex_numeric = 1 (male), age_numeric = 33, and generation_numeric = 7, representing Generation Alpha as the generation following Generation Z in the ordinal encoding scheme. The remaining predictors (year, population, gdp_for_year , and gdp_per_capita ) were set to their mean values from the dataset to provide baseline conditions for the prediction.

This prediction illustrates how the Q3 model can extrapolate to unseen categories because generation was represented as a numeric variable rather than separate categorical indicators. As discussed in the regression chapter of the textbook, linear regression models estimate outcomes based on linear combinations of input features, which allows predictions for new numeric values even if they were not present in the training data (Raschka, 2026, Ch. 9).

#### Q7: [10 pts] Give one  advantage  when using  regression  (as opposed  to classification  with nominal  features) in terms of independent  variables.

One advantage of using regression instead of classification with nominal features is that regression can naturally model continuous independent variables and their quantitative relationships with the target variable. In a regression model, predictors such as age, population, or economic indicators can be used directly as numerical inputs, allowing the model to estimate how incremental changes in those variables affect the predicted outcome. This preserves the underlying structure of the data and allows the model to capture trends and relationships across a continuous range of values. In contrast, classification approaches using nominal features often require discretizing variables into categories, which can remove useful information about the magnitude and direction of relationships between predictors and the outcome.

#### Q8: [10 pts] Give one  advantage  when using  regular  numerical values rather  than one-hot encoding  for regression

One advantage of using regular numerical values instead of one-hot encoding in regression is that it reduces the number of features in the model, making the model simpler and more efficient to train. When categorical variables are one-hot encoded, each category becomes its own feature, which can significantly increase the dimensionality of the dataset. In contrast, representing variables with numerical values allows a single feature to capture the information, reducing the number of parameters the model must estimate. This can improve computational efficiency and make the model easier to interpret, since each coefficient directly represents how changes in the numeric predictor affect the target variable.

#### Q9: [10 pts] Now that you developed  both  a classifier  (previously)  and a regression  model for the problem in this  assignment,  which method do you suggest  to your machine learning model customer? Classifier  or regression?  Why? 


After developing both a classifier in Module 3 and regression models in this assignment, I would recommend using a regression model for this problem. In Module 3 the classification approach required converting the continuous target variable suicides per 100k population into discrete categories such as high and low suicide rates based on a threshold. While the classifier achieved reasonable performance of about 77 percent accuracy, this approach simplified the problem and removed information about the true magnitude of the suicide rate.

In contrast, the regression models developed in this assignment directly predict the continuous suicide rate and allow the model to capture more detailed relationships between demographic and economic predictors and the outcome. Regression models also produce interpretable coefficients that show how strongly each feature contributes to the predicted value. From a public health perspective this is especially valuable because identifying variables with larger coefficients such as age or economic indicators can help highlight potential risk factors and guide prevention strategies or policy decisions. Large public health and economic data platforms such as the World Bank Human Capital Project commonly rely on regression and correlation based analyses to understand relationships between economic and health indicators because these methods preserve the continuous nature of the data and provide quantitative insight into how predictors influence outcomes (World Bank, 2020). In addition, regression methods are well suited for modeling continuous outcomes and interpreting the influence of predictors on those outcomes (Raschka, 2026, Ch. 9).

#### References


Raschka, S., Liu, Y., & Mirjalili, V. (2022). Machine learning with pytorch and scikit-learn : develop machine learning and deep learning models with python. Packt Publishing.

WorldBank. (2026, January 11). Human Capital Index (HCI) (scale 0-1). Human capital indicator. https://humancapital.worldbank.org/en/indicator/WB_HCP_HCI?view=Correlation&year=2020#explore-indicator 